Prova interpolazione traiettoria nei punti di curva

In [1]:
import os

import h5py

import numpy as np
import pandas as pd

from bokeh.models import ColumnDataSource, DataRange1d, HoverTool, LinearAxis
from bokeh.palettes import d3
from bokeh.plotting import figure, show
from bokeh.io import output_notebook

import matplotlib.pyplot as plt

output_notebook()

Loading BokehJS ...

In [21]:
data_folder = './' 
segment_id = 81222
data_filename = 'segment_{:d}.h5'.format(segment_id)

In [22]:


f = h5py.File(data_filename, 'r')
f.visit(print)

athlete
processed
processed/com
processed/com/gnss_time_offset
processed/com/inclination
processed/com/p_antenna_com
processed/com/pos
processed/com/pos_offset
processed/com/q_sensor
processed/com/timestamp
processed/com/turn_radius
processed/com/vel
processed/turn_stats
processed/turn_stats/turn_switches
sensor
sensor/bar
sensor/bar/pres
sensor/bar/temp
sensor/bar/timestamp
sensor/gnss
sensor/gnss/day
sensor/gnss/fix_type
sensor/gnss/flags
sensor/gnss/flags2
sensor/gnss/flags3
sensor/gnss/ground_speed
sensor/gnss/heading
sensor/gnss/heading_acc
sensor/gnss/heading_vehicule
sensor/gnss/height_ellipsoid
sensor/gnss/height_msl
sensor/gnss/horizontal_acc
sensor/gnss/hour
sensor/gnss/itow
sensor/gnss/lat
sensor/gnss/lon
sensor/gnss/mag_acc
sensor/gnss/mag_dec
sensor/gnss/min
sensor/gnss/month
sensor/gnss/nano
sensor/gnss/nb_sats
sensor/gnss/position_dop
sensor/gnss/sec
sensor/gnss/speed_acc
sensor/gnss/time_acc
sensor/gnss/timestamp
sensor/gnss/valid
sensor/gnss/vel_down
sensor/gnss/vel_ea

In [23]:
with h5py.File(os.path.join(data_folder, data_filename), 'r') as data:

    p_com = data['processed/com/pos'][:, :]
    p_com = p_com[1200:]
    pos_offset = data['processed/com/pos_offset'][:]

    print(data['processed'].keys())

    turn_switch_inds = data['processed/turn_stats/turn_switches'][:].astype(int)
    
#p_com += pos_offset
turn_switch_inds = turn_switch_inds - 1200

<KeysViewHDF5 ['com', 'turn_stats']>


In [24]:
print(type(turn_switch_inds))
print(turn_switch_inds.dtype)
print(turn_switch_inds.shape)
print(turn_switch_inds[:10])
print(p_com[0:20])

<class 'numpy.ndarray'>
int64
(12,)
[ 399  675  993 1336 1771 2127 2635 3053 3500 3902]
[[-8.67156787 -4.92576253 -2.3780612 ]
 [-8.69493363 -4.92988456 -2.38449918]
 [-8.71834327 -4.93401279 -2.390969  ]
 [-8.7417949  -4.93814632 -2.3974704 ]
 [-8.76528638 -4.94228312 -2.40400256]
 [-8.78881394 -4.94641433 -2.41056547]
 [-8.8123771  -4.95053962 -2.4171585 ]
 [-8.83597589 -4.95465811 -2.42378076]
 [-8.85961196 -4.95876878 -2.43043269]
 [-8.88328717 -4.96287061 -2.43711622]
 [-8.90700141 -4.96696235 -2.44383281]
 [-8.93075305 -4.97104298 -2.45058264]
 [-8.9545406  -4.975112   -2.45736479]
 [-8.97836425 -4.97916936 -2.46417816]
 [-9.00222658 -4.98321554 -2.47102254]
 [-9.02613074 -4.98725156 -2.47789851]
 [-9.05007987 -4.99127893 -2.48480746]
 [-9.07407526 -4.99529929 -2.49175101]
 [-9.09811499 -4.99931401 -2.49872932]
 [-9.1221975  -5.00332471 -2.50574098]]


In [29]:
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
import numpy as np

indices = np.arange(len(p_com))
turn_interp_ind = [0, 470, 808, 1181, 1580, 2038, 2384, 2858, 3303, 3770, 4153, 4485, 4698]

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=p_com[:, 0],
    y=p_com[:, 1],
    z=p_com[:, 2],
    mode='lines',
    name='p_com',
    hovertext=indices,

    hovertemplate=
    "index: %{hovertext}<br>" +
    "x: %{x}<br>" +
    "y: %{y}<br>" +
    "z: %{z}<extra></extra>"
))

fig.add_trace(go.Scatter3d(
    x=p_com[turn_interp_ind,0],
    y=p_com[turn_interp_ind,1],
    z=p_com[turn_interp_ind,2],
    mode='markers',
    name='Turn points',
    marker=dict(
        size=6,
        color='red'
    )
))

fig.update_layout(
    title='Traiettoria del centro di massa',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data'
    )
)

fig.show()

Inizio interpolazione passante per turn_interp_ind

In [30]:
turn_interp_ind = np.array(turn_interp_ind)

interp_coord = p_com[turn_interp_ind]

In [33]:
print(interp_coord[0::])

[[  -8.67156787   -4.92576253   -2.3780612 ]
 [ -24.78341483   -3.88378177   -7.67819282]
 [ -38.16090035   -9.03933943  -12.67143888]
 [ -54.89560327   -6.99280196  -17.79481597]
 [ -70.14984458  -17.48054162  -24.25489428]
 [ -93.92767845  -14.35050908  -31.94442079]
 [-113.7527781   -19.40241194  -39.19059605]
 [-137.50079358   -1.71816561  -47.47915124]
 [-167.00768835    1.70471837  -58.98800375]
 [-194.73020157   24.58168854  -70.22050041]
 [-224.77392591   34.24792972  -78.41635452]
 [-249.93439359   45.29098998  -87.77577057]
 [-264.77046747   48.42851892  -91.20398642]]


In [38]:
from scipy.interpolate import splprep, splev
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

tck, u = splprep(interp_coord.T, s=0, k = 3)
u_fine = np.linspace(0, 1, 200)
x_fine, y_fine, z_fine = splev(u_fine, tck)

In [39]:
# ---- Create figure ----
fig = go.Figure()

# Add curve
fig.add_trace(go.Scatter3d(
    x=p_com[:, 0],
    y=p_com[:, 1],
    z=p_com[:, 2],
    mode='lines',
    name='p_com',
    hovertext=indices,
))

fig.add_trace(go.Scatter3d(
    x=x_fine,
    y=y_fine,
    z=z_fine,
    mode='lines',
    line=dict(width=4),
    name="Rec"
))

fig.add_trace(go.Scatter3d(
    x=p_com[turn_interp_ind,0],
    y=p_com[turn_interp_ind,1],
    z=p_com[turn_interp_ind,2],
    mode='markers',
    name='Turn points',
    marker=dict(
        size=6,
        color='red'
    )
))


fig.update_layout(
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    )
)

fig.show()